# GNN-Based BERT for Understanding Context from Music — Demo

This notebook demonstrates the **completed project artifacts and trained-model inference pipeline**.

It is designed to be run from the project root:

```text
gnn_bert_music/
├── data/
├── models/
├── results/
├── src/
└── notebooks/
```

The notebook has two purposes:

1. **Reproduce the final evaluation summary** from the saved result files.
2. **Run live inference** on a selected test track using the saved BERT, GNN, early-fusion, and cross-attention model weights, when those weights are present.

The final experiment predicts 20 music tags from a leakage-free textual context (title, artist, album, genre) and audio-derived graph features.


In [ ]:
# If necessary, install the project dependencies first:
# pip install -r requirements.txt

from pathlib import Path
import sys
import json
import ast
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from config import (
    SPLITS_DIR,
    PROCESSED_DIR,
    RESULTS_DIR,
    MODELS_DIR,
    BERT_MODEL_NAME,
    MAX_TEXT_LENGTH,
    NUM_LABELS,
)
from utils import get_device

DEVICE = get_device()

print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)


## 1. Load the final dataset and label vocabulary

In [ ]:
labels_path = SPLITS_DIR / "labels.json"
dataset_path = SPLITS_DIR / "context_dataset.csv"

with open(labels_path, "r", encoding="utf-8") as f:
    labels = json.load(f)

df = pd.read_csv(dataset_path)

print("Number of labels:", len(labels))
print("Labels:", labels)
print()
print("Dataset shape:", df.shape)
print(df["split"].value_counts())
display(df.head())


## 2. Final test-set results

These are the final reported metrics from the completed experiments. The primary F1 scores use a fixed threshold of 0.5.


In [ ]:
comparison_path = RESULTS_DIR / "final_comparison.csv"
comparison = pd.read_csv(comparison_path)

display(comparison.style.format({
    "Macro-F1": "{:.4f}",
    "Micro-F1": "{:.4f}",
    "Macro AUC-PR": "{:.4f}",
    "Micro AUC-PR": "{:.4f}",
}))


### Main result

The cross-attention fusion model has the strongest **Macro-F1**, **Macro AUC-PR**, and **Micro AUC-PR**. Context BERT has the strongest **Micro-F1**.


## 3. Inspect one test track

In [ ]:
test_df = df[df["split"] == "test"].reset_index(drop=True)

# Change this index to inspect another test track.
TEST_INDEX = 0

row = test_df.iloc[TEST_INDEX]
track_id = int(row["track_id"])

true_vector = np.asarray(ast.literal_eval(row["labels"]), dtype=np.float32)
true_labels = [labels[i] for i, value in enumerate(true_vector) if value > 0]

print("Track ID:", track_id)
print("Context:", row["text"])
print("Genre:", row["genre"])
print("True labels:", true_labels)

graph_path = PROCESSED_DIR / "graphs" / f"{track_id:06d}.pt"
graph = torch.load(graph_path, weights_only=False)

print()
print("Graph nodes:", graph.x.shape[0])
print("Node feature dimension:", graph.x.shape[1])
print("Edges:", graph.edge_index.shape[1])


## 4. Show saved predictions for this track

The results archive contains predictions generated on the final test set. This section lets the demo work even if model checkpoints are not copied into the project folder yet.


In [ ]:
prediction_files = {
    "Majority": RESULTS_DIR / "predictions" / "majority_predictions.npz",
    "CNN": RESULTS_DIR / "predictions" / "cnn_predictions.npz",
    "GNN": RESULTS_DIR / "predictions" / "gnn_predictions.npz",
    "Context BERT": RESULTS_DIR / "predictions" / "context_bert_predictions.npz",
    "Early Fusion": RESULTS_DIR / "predictions" / "fusion_predictions.npz",
    "Cross-Attention": RESULTS_DIR / "predictions" / "cross_attention_predictions.npz",
}

def load_prediction_file(path):
    data = np.load(path)
    # The saved arrays contain probabilities and labels; identify them by common names.
    return {k: data[k] for k in data.files}

def top_predictions(probabilities, labels, k=5):
    probabilities = np.asarray(probabilities).reshape(-1)
    order = np.argsort(probabilities)[::-1][:k]
    return [(labels[i], float(probabilities[i])) for i in order]

for name, path in prediction_files.items():
    if not path.exists():
        print(name, "prediction file not found:", path)
        continue

    data = load_prediction_file(path)
    print(name, "arrays:", list(data.keys()))

    # Most saved files contain a probability array. Try common names.
    prob_key = next((k for k in ["probabilities", "probs", "predictions"] if k in data), None)
    if prob_key is None:
        print("  Could not automatically identify probability array.")
        continue

    probs = data[prob_key]

    # Find the row corresponding to this test index when the array is aligned with test_df.
    if len(probs) == len(test_df):
        print("  Top 5:", top_predictions(probs[TEST_INDEX], labels))
    else:
        print("  Prediction array shape:", probs.shape)

    print()


## 5. Optional live inference with saved checkpoints

The following section loads the project's trained model classes and performs a real forward pass.

Expected checkpoint names:

```text
models/context_bert_model.pt
models/gnn_model.pt
models/fusion_model.pt
models/cross_attention_model.pt
```

If your checkpoint filenames differ, change the paths below. The section is optional; the saved prediction section above is the artifact-only demonstration.


In [ ]:
from transformers import AutoTokenizer

from bert_model import BERTTagClassifier
from gnn_model import MusicGraphSAGE
from fusion_model import FusionModel
from cross_attention_model import CrossAttentionFusionModel
from fusion_dataset import FusionDataset

tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)

def find_checkpoint(*names):
    for name in names:
        path = MODELS_DIR / name
        if path.exists():
            return path
    return None

bert_ckpt = find_checkpoint("context_bert_model.pt", "bert_model.pt")
gnn_ckpt = find_checkpoint("gnn_model.pt")
fusion_ckpt = find_checkpoint("fusion_model.pt")
cross_ckpt = find_checkpoint("cross_attention_model.pt")

print("BERT checkpoint:", bert_ckpt)
print("GNN checkpoint:", gnn_ckpt)
print("Early-fusion checkpoint:", fusion_ckpt)
print("Cross-attention checkpoint:", cross_ckpt)


In [ ]:
# Prepare one text example.
encoding = tokenizer(
    str(row["text"]),
    padding="max_length",
    truncation=True,
    max_length=MAX_TEXT_LENGTH,
    return_tensors="pt"
)

input_ids = encoding["input_ids"].to(DEVICE)
attention_mask = encoding["attention_mask"].to(DEVICE)

graph_x = graph.x.to(DEVICE)
edge_index = graph.edge_index.to(DEVICE)

def load_state(model, checkpoint):
    state = torch.load(checkpoint, map_location=DEVICE, weights_only=True)
    # Support either a raw state_dict or a training dictionary containing model_state_dict.
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]
    model.load_state_dict(state)
    model.to(DEVICE)
    model.eval()
    return model

@torch.no_grad()
def predict_model(model, model_type):
    if model_type == "bert":
        logits = model(input_ids, attention_mask)
    elif model_type == "gnn":
        logits = model(graph_x, edge_index)
    elif model_type == "fusion":
        logits = model(input_ids, attention_mask, graph_x, edge_index)
    elif model_type == "cross":
        logits = model(input_ids, attention_mask, graph_x, edge_index)
    else:
        raise ValueError(model_type)
    return torch.sigmoid(logits).detach().cpu().numpy().reshape(-1)

live_predictions = {}

if bert_ckpt:
    model = load_state(BERTTagClassifier(BERT_MODEL_NAME, NUM_LABELS), bert_ckpt)
    live_predictions["Context BERT"] = predict_model(model, "bert")

if gnn_ckpt:
    model = load_state(MusicGraphSAGE(25, 64, NUM_LABELS), gnn_ckpt)
    live_predictions["GNN"] = predict_model(model, "gnn")

if fusion_ckpt:
    model = load_state(FusionModel(BERT_MODEL_NAME, NUM_LABELS, 25, 64), fusion_ckpt)
    live_predictions["Early Fusion"] = predict_model(model, "fusion")

if cross_ckpt:
    model = load_state(CrossAttentionFusionModel(BERT_MODEL_NAME, NUM_LABELS, 25, 64, 8), cross_ckpt)
    live_predictions["Cross-Attention"] = predict_model(model, "cross")

print("Live models successfully evaluated:", list(live_predictions))


In [ ]:
# Display live top-5 predictions.
for name, probs in live_predictions.items():
    print(f"\n{name}")
    for label, probability in top_predictions(probs, labels, k=5):
        print(f"  {label:20s} {probability:.4f}")


## 6. Reproduce the main plots

The project stores the completed comparison plots in `results/plots/`. This cell displays them inside the notebook.


In [ ]:
from IPython.display import Image, display

plot_names = [
    "macro_f1_comparison.png",
    "micro_f1_comparison.png",
    "macro_auc_pr_comparison.png",
    "micro_auc_pr_comparison.png",
    "cross_attention_loss.png",
    "cross_attention_validation_f1.png",
]

for plot_name in plot_names:
    path = RESULTS_DIR / "plots" / plot_name
    if path.exists():
        print(plot_name)
        display(Image(filename=str(path)))


## 7. Notes on interpretation

- The final supervised dataset contains **393 training, 44 validation, and 50 test tracks**.
- The target vocabulary contains **20 labels**.
- The graph uses **5-second segments**, **13 MFCC + 12 chroma = 25 node features**, temporal edges, and cosine-similarity edges with threshold 0.8.
- The final primary test metrics use a **0.5 threshold**.
- The exploratory test-set threshold sweep is **not** used for the official result because tuning a threshold on the test set would bias evaluation.
- An earlier experiment that fed target tags into BERT was discarded as **target leakage**.
- The optional MusicCaps/InfoNCE retrieval extension was not implemented in the completed experiment.
